# Getting nse symbols

In [ ]:
import sys
sys.path.append("C:/Users/kashi/python/ib/src")

In [ ]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

def get_nse_syms() -> pd.DataFrame:
    """Generates symbols for nse with expiry months having lots"""

    url = "https://www1.nseindia.com/content/fo/fo_mktlots.csv"

    try:
        req = requests.get(url)
        if req.status_code == 404:
            print(f"\n{url} URL contents not correct. 404 error!")
        df_syms = pd.read_csv(StringIO(req.text))
    except requests.ConnectionError as e:
        print(f"Connection Error {e}")
    except pd.errors.ParserError as e:
        print(f"Parser Error {e}")

    df_syms = df_syms[list(df_syms)[1:5]] 

    # strip whitespace from columns and make it lower case
    df_syms.columns = df_syms.columns.str.strip().str.lower()

    # strip all string contents of whitespaces
    df_syms = df_syms.applymap(lambda x: x.strip()
                                        if type(x) is str else x)

    # remove 'Symbol' row
    df_syms = df_syms[df_syms.symbol != "Symbol"]

    # introduce `secType`
    df_syms.insert(1, 'secType', np.where(df_syms.symbol.str.contains('NIFTY'), 'IND', 'STK'))

    # introduce `exchange`
    df_syms.insert(2, 'exchange', 'NSE')

    # introduce `currency`
    df_syms.insert(3, 'currency', 'INR')

    return df_syms

In [ ]:
df_syms = get_nse_syms()
df_syms.head(5)

# Get Option Chain

In [ ]:
symbol = 'NIFTY'

from nsepython import nse_optionchain_scrapper, nse_get_fno_lot_sizes
import datetime
import pandas as pd


def get_nse_opts(symbol: str) -> pd.DataFrame:
        """Get Option Chains for a symbol"""

        scraped = nse_optionchain_scrapper(symbol)

        raw_dict = scraped['records']['data']
        dfs = [pd.DataFrame.from_dict(raw_dict[i]).transpose()[2:] 
                for i in range(len(raw_dict))]

        df = pd.concat(dfs).drop(columns='identifier').rename_axis('right').reset_index()

        float_cols = ['strikePrice', 'pchangeinOpenInterest', 'impliedVolatility', 
                'lastPrice', 'change', 'pChange', 'bidprice', 'askPrice' , 
                'underlyingValue' ]
        int_cols = ['openInterest', 'changeinOpenInterest', 'totalTradedVolume', 
                'totalBuyQuantity', 'totalSellQuantity', 'bidQty', 'askQty']
        df[float_cols] = df[float_cols].astype('float32')
        df[int_cols] = df[int_cols].astype('int64')
        df['expiryDate'] = pd.to_datetime(df.expiryDate)

        colmap = { 'underlying': 'symbol', 'expiryDate': 'expiry', 'strikePrice': 'strike',  
                'right': 'right','underlyingValue': 'undPrice', 'openInterest': 'oi',
                'changeinOpenInterest': 'oiChange', 'pchangeinOpenInterest': 'pChangeOI',
                'totalTradedVolume': 'volume', 'totalBuyQuantity': 'totalBuyQty', 
                'totalSellQuantity': 'totalSellQty', 'pChange': 'pChange', 
                'impliedVolatility': 'opt_iv',   'lastPrice': 'lastPrice', 'change': 'change',
                'bidQty': 'bidQty', 'bidprice': 'bid', 'askPrice': 'ask', 'askQty': 'askQty',
                }

        df = df.rename(columns=colmap)[colmap.values()]
        df = df.assign(lot=nse_get_fno_lot_sizes(symbol=symbol))
        df = df.assign(opt_iv = df.opt_iv/100)
        df = df.assign(right=df.right.str[:1])
        df = df.sort_values(['expiry', 'right', 'strike'], 
                        ascending=[True, False, True]).reset_index(drop=True)

        return df



In [ ]:
import pytz
import datetime



# ..get accurate dte
nse_tz = pytz.timezone('Asia/Kolkata')
now = datetime.datetime.now(tz=nse_tz).timestamp()
nse_tz_expiry = df.expiry.apply(lambda x: nse_tz.localize(datetime.datetime.combine(x.date(), datetime.time(18,0))).timestamp())
dte = (nse_tz_expiry.apply(datetime.datetime.fromtimestamp) - datetime.datetime.fromtimestamp(now)).apply(datetime.timedelta.total_seconds)/24/3600
df = df.assign(dte=dte)

In [ ]:
from ib_insync import IB, Contract


import numpy as np


In [ ]:

def nse():

    # Get the symbols
    url = "https://www1.nseindia.com/content/fo/fo_mktlots.csv"

    try:
        req = requests.get(url)
        if req.status_code == 404:
            print(f"\n{url} URL contents not correct. 404 error!")
        df_symlots = pd.read_csv(StringIO(req.text))
    except requests.ConnectionError as e:
        print(f"Connection Error {e}")
    except pd.errors.ParserError as e:
        print(f"Parser Error {e}")

    df_symlots = df_symlots[list(df_symlots)[1:5]] 

    # strip whitespace from columns and make it lower case
    df_symlots.columns = df_symlots.columns.str.strip().str.lower()

    # strip all string contents of whitespaces
    df_symlots = df_symlots.applymap(lambda x: x.strip()
                                        if type(x) is str else x)

    # remove 'Symbol' row
    df_symlots = df_symlots[df_symlots.symbol != "Symbol"]

    # melt the expiries into rows
    df_symlots = df_symlots.melt(id_vars=["symbol"],
                                    var_name="expiryM",
                                    value_name="lot").dropna()

    # remove rows without lots
    df_symlots = df_symlots[~(df_symlots.lot == "")]


    # convert expiry to period
    df_symlots = df_symlots.assign(expiryM=pd.to_datetime(
        df_symlots.expiryM.apply(lambda x: x.title()), 
                            format="%b-%y")\
                                .dt.to_period("M").astype("str"))


    # convert lots to integers
    df_symlots = df_symlots.assign(
        lot=pd.to_numeric(df_symlots.lot, errors="coerce"))


    # convert & to %26
    df_symlots = df_symlots.assign(
        symbol=df_symlots.symbol.str.replace("&", ""))

    # make it 9 chars
    df_symlots = df_symlots.assign(symbol=df_symlots.symbol.str.slice(0, 9))

    # remove FINNIFTY
    df_symlots = df_symlots[~df_symlots.symbol.str.contains('FINNIFTY')]
    df_symlots.symbol.replace({'NIFTY': 'NIFTY50'}, inplace=True)

    # differentiate between index and stock
    df_symlots.insert(
        1, "secType",
        np.where(df_symlots.symbol.str.contains("NIFTY"), "IND", "STK"))

    df_symlots["exchange"] = "NSE"
    df_symlots["currency"] = "INR"
    df_symlots["contract"] = [
        Contract(symbol=symbol,
                    secType=secType,
                    exchange=exchange,
                    currency=currency)
        for symbol, secType, exchange, currency in zip(
            df_symlots.symbol,
            df_symlots.secType,
            df_symlots.exchange,
            df_symlots.currency,
        )
    ]

    # get the qualified contracts
    contracts = df_symlots.groupby('symbol').first().contract

    # contracts = qualify(ib, cts, msg='unds qual')
        
    df_symlots = df_symlots.assign(contract=df_symlots.symbol.\
                        map({c.symbol: c for c in contracts})).\
                            dropna(subset=['contract'])

    return df_symlots

In [ ]:
df = nse()
df